In [1]:
!pip install datasets
!pip install ragas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.8/174.8 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 117.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.8/358.8 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.4/226.4 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%%bash
sudo apt-get update
sudo apt-get install -y zstd

curl -fsSL https://ollama.com/install.sh | sh

# Ensure the base Ollama config directory exists
mkdir -p ~/.ollama

# Create a persistent models directory in Google Drive
mkdir -p /content/drive/MyDrive/RAG/ollama_models

# Remove any existing 'models' directory or symlink within ~/.ollama
rm -rf ~/.ollama/models

# Symlink Ollama's model directory to Google Drive
ln -s /content/drive/MyDrive/RAG/ollama_models ~/.ollama/models

echo 'Symlinked ~/.ollama/models -> /content/drive/MyDrive/RAG/ollama_models'

# Adding Ollama to PATH if it's not already, though the installer should handle this
export PATH="$PATH:/usr/local/bin"

# Start ollama serve in the background and detach it from the shell
nohup ollama serve > ollama_serve.log 2>&1 &

# Give it a few seconds to start up
sleep 10



Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,297 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,868 kB]
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:114

In [4]:
# Now, pull the model
!ollama pull gemma3:27b

!ollama pull nomic-embed-text

In [5]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings import embedding_factory

aclient = AsyncOpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)

llm = llm_factory(
    provider="openai",
    model="gemma3:27b",
    client=aclient,
    temperature=0,
    max_tokens=100000,
)

embeddings = embedding_factory(
    provider="openai",
    model="nomic-embed-text",
    client=aclient,
)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipython-input-1227768394.py:18: DeprecationWarning: Importing embedding_factory from ragas.embeddings is deprecated. Import directly from ragas.embeddings.base or use modern providers: from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  embeddings = embedding_factory(


In [6]:
from datasets import Dataset
import pandas as pd

d1_response = {
    'question': ['When was the official directive emphasized to use school textbooks, software and other educational tools in classrooms in French Belgium ?'],
    'answer': ['The official directive was emphasized in **2006**.'],
    'ground_truth': ['2006']
}

g1_response = {
    'question': ['When was the official directive emphasized to use school textbooks, software and other educational tools in classrooms in French Belgium ?'],
    'answer': ['''Okay, let's break this down step-by-step to answer the question.

1.  **Identify the question:** The question asks about the year the official directive emphasizing the use of school textbooks, software, and other educational tools was issued in French Belgium.

2.  **Scan the context:** I need to carefully read the provided text to find the answer.

3.  **Locate relevant information:** The text states: "A 2006 official directive emphasizes the use of school textbooks, software, and other educational tools in classrooms."

4.  **Answer the question:** Therefore, the official directive was emphasized in 2006.

**Answer:** 2006'''],
    'ground_truth': ['2006']
}

l1_response = {
    'question': ['When was the official directive emphasized to use school textbooks, software and other educational tools in classrooms in French Belgium ?'],
    'answer': ['''To answer this question, I will walk you through my thought process step by step.

Step 1: Read the context carefully to identify any relevant information.
The context provided is about Belgium (French) and its education system. It mentions a 2006 official directive that emphasizes the use of school textbooks, software, and other educational tools in classrooms.

Step 2: Identify the specific sentence or phrase that answers the question.
The sentence that answers the question is: "A 2006 official directive emphasizes the use of school textbooks, software, and other educational tools in classrooms."

Step 3: Extract the relevant information from the sentence.
According to this sentence, the official directive was emphasized in 2006.

Therefore, my answer is:

The official directive emphasizing the use of school textbooks, software, and other educational tools in classrooms in French Belgium was in 2006.'''],
    'ground_truth': ['2006']
}

d2_response = {
    'question': ['When was the official directive emphasized to use school textbooks, software and other educational tools in classrooms in French Belgium ?'],
    'answer': ['The two important educational decrees were promulgated in **1995** and **1997**.'],
    'ground_truth': ['2006']
}

g2_response = {
    'question': ['When was the official directive emphasized to use school textbooks, software and other educational tools in classrooms in French Belgium ?'],
    'answer': ['Two important educational degrees were promulgated in French Belgium during the late 1990s. In **1995**, a decree was issued aimed at promoting a successful school in primary education. Subsequently, in **1997**, a further decree defined the priority missions for both primary and secondary education. These two decrees represent key milestones in shaping the educational landscape of French-speaking Belgium at that time.'],
    'ground_truth': ['2006']
}

l2_response = {
    'question': ['When was the official directive emphasized to use school textbooks, software and other educational tools in classrooms in French Belgium ?'],
    'answer': ['''Based on the provided information, I infer that two important educational degrees were promulgated in French Belgium,
    specifically in the French-speaking Community of Belgium, around the mid-to-late 1990s. The decrees mentioned (Gouvernement de la Communauté
    française, 1997 and Gouvernement de la Communauté française, 1995) likely outline policies or regulations for education at a primary or secondary
    level. However, without explicit mention of degree names, this answer is based on general knowledge and inference from the provided documents. The
    exact dates are not specified but fall within the late 1990s timeframe.'''],
    'ground_truth': ['2006']
}

dataset_d1 = Dataset.from_dict(d1_response)
dataset_g1 = Dataset.from_dict(g1_response)
dataset_l1 = Dataset.from_dict(l1_response)
dataset_d2 = Dataset.from_dict(d2_response)
dataset_g2 = Dataset.from_dict(g2_response)
dataset_l2 = Dataset.from_dict(l2_response)

In [7]:
import nest_asyncio

from ragas import evaluate
from ragas.metrics import ContextPrecision, ContextRecall, Faithfulness, AnswerCorrectness
from ragas.run_config import RunConfig
nest_asyncio.apply()

run_config = RunConfig(max_workers=5, timeout=600)

metrics = [
    AnswerCorrectness(llm=llm),
]


/tmp/ipython-input-4227088422.py:4: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import ContextPrecision, ContextRecall, Faithfulness, AnswerCorrectness
/tmp/ipython-input-4227088422.py:4: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextRecall
  from ragas.metrics import ContextPrecision, ContextRecall, Faithfulness, AnswerCorrectness
/tmp/ipython-input-4227088422.py:4: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import ContextPrecision, Conte

In [8]:
results_d1 = evaluate(
    dataset=dataset_d1,
    metrics=metrics,
    llm=llm,
    embeddings=embeddings,
    raise_exceptions=False,
    run_config=run_config,
)

print(results_d1)

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

{'answer_correctness': 0.6843}


In [9]:
results_l1 = evaluate(
    dataset=dataset_l1,
    metrics=metrics,
    llm=llm,
    embeddings=embeddings,
    raise_exceptions=False,
    run_config=run_config,
)

print(results_l1)

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

{'answer_correctness': 0.6248}


In [10]:
results_g1 = evaluate(
    dataset=dataset_g1,
    metrics=metrics,
    llm=llm,
    embeddings=embeddings,
    raise_exceptions=False,
    run_config=run_config,
)

print(results_g1)

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

{'answer_correctness': 0.5596}


In [11]:
results_l2 = evaluate(
    dataset=dataset_l2,
    metrics=metrics,
    llm=llm,
    embeddings=embeddings,
    raise_exceptions=False,
    run_config=run_config,
)

print(results_l2)

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

{'answer_correctness': 0.1048}


In [12]:
results_g2 = evaluate(
    dataset=dataset_g2,
    metrics=metrics,
    llm=llm,
    embeddings=embeddings,
    raise_exceptions=False,
    run_config=run_config,
)

print(results_g2)

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

{'answer_correctness': 0.1070}


In [13]:
results_d2 = evaluate(
    dataset=dataset_d2,
    metrics=metrics,
    llm=llm,
    embeddings=embeddings,
    raise_exceptions=False,
    run_config=run_config,
)

print(results_d2)

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

{'answer_correctness': 0.1098}


In [14]:
results_combined = {'results_l1': results_l1,
                    'results_g1': results_g1,
                    'results_d1': results_d1,
                    'results_l2': results_l2,
                    'results_g2': results_g2,
                    'results_d2': results_d2}

In [15]:
for key, value in results_combined.items():
  print(key)
  print(value)
  print('\n')

results_l1
{'answer_correctness': 0.6248}


results_g1
{'answer_correctness': 0.5596}


results_d1
{'answer_correctness': 0.6843}


results_l2
{'answer_correctness': 0.1048}


results_g2
{'answer_correctness': 0.1070}


results_d2
{'answer_correctness': 0.1098}


